# Test Depth Anything V2 with your computer's webcam (Mac and Windows)

This notebook runs live depth estimation from your built-in webcam. You see your camera feed on the left and the predicted depth map on the right in a separate window.

**Before you start:** finish the setup notebook first. You need the `Depth-Anything-V2` repo cloned, its dependencies installed, and a checkpoint in `checkpoints/` (for example `depth_anything_v2_vits.pth`).

The video opens in its own window, so run this notebook locally (Jupyter or VS Code on your own computer). It will not work in a hosted notebook such as Google Colab, because those cannot reach your webcam.

## Step 1: Allow camera access

- **Mac:** the first run shows a camera permission prompt for the app running the notebook (Terminal, VS Code, etc.). Click **Allow**. If you missed it, go to **System Settings > Privacy & Security > Camera**, turn on that app, then restart it.
- **Windows:** go to **Settings > Privacy & security > Camera**. Turn on **Camera access** and **Let desktop apps access your camera**. Close other apps that use the webcam (Zoom, Teams, browsers).

## Step 2: Move into the repo folder

Run this only if the notebook is not already inside `Depth-Anything-V2`. Change the path to wherever you cloned the repo.

In [ ]:
import os

# Adjust if needed, for example '../Depth-Anything-V2'
if not os.path.exists('depth_anything_v2'):
    %cd Depth-Anything-V2

print(os.getcwd())
print(os.listdir('checkpoints'))

## Step 3: Create the webcam script

This cell writes `webcam_depth.py` into the repo folder. Edit `ENCODER` so it matches your checkpoint (`vits`, `vitb` or `vitl`).

In [ ]:
%%writefile webcam_depth.py
import platform
import sys
import time

import cv2
import matplotlib
import numpy as np
import torch

from depth_anything_v2.dpt import DepthAnythingV2

MODEL = 'teacher'      # 'teacher' = Depth Anything V2, 'student' = Level 2 U-Net (train it in Level2-Distill-Depth.ipynb).
ENCODER = 'vits'       # 'vits', 'vitb' or 'vitl'. Must match your checkpoint.
CAMERA_INDEX = 0       # 0 = default webcam. Try 1 if the wrong camera opens.
INPUT_SIZE = 308       # Lower = faster, higher = more detail. The default in the repo is 518.
BLUR_KSIZE = 15        # Gaussian blur size (odd) before finding NEAR/FAR. Bigger = steadier, but thin objects (a finger) fade.
SMOOTH_ALPHA = 0.3     # Weight of the newest frame in the running average of the depth map. Lower = steadier but laggier.
FAR_PERCENT = 2        # FAR = centre of the largest patch among the farthest X% of pixels. Bigger = steadier, less precise.
GRID_ROWS, GRID_COLS = 6, 8  # Obstacle grid drawn on the camera feed. Set GRID_ROWS = 0 to hide it.
GRID_STAT = 'mean'     # 'mean' = average closeness per cell (steady); 'max' = closest pixel per cell (won't hide a thin pole).
EDGE_MARGIN = 0.05     # Fraction of width/height ignored on each border when searching. 0 = search the whole frame.

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Using device:', DEVICE)

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64,  'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}

if MODEL == 'student':
    from student_unet import StudentDepth  # written by Level2-Distill-Depth.ipynb; same infer_image() call
    model = StudentDepth('checkpoints/student_unet.pth', DEVICE)
else:
    model = DepthAnythingV2(**model_configs[ENCODER])
    model.load_state_dict(torch.load(f'checkpoints/depth_anything_v2_{ENCODER}.pth', map_location='cpu'))
    model = model.to(DEVICE).eval()

# On Windows the DirectShow backend opens the webcam faster and more reliably.
if platform.system() == 'Windows':
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
else:
    cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    sys.exit('Could not open the webcam. Check camera permissions and that no other app is using it.')

cmap = matplotlib.colormaps.get_cmap('Spectral_r')


def region_center(region):
    """Centre (x, y) of the largest connected patch in a boolean mask."""
    _, _, stats, centers = cv2.connectedComponentsWithStats(region.astype(np.uint8))
    biggest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])  # label 0 is the background
    return int(centers[biggest][0]), int(centers[biggest][1])


def depth_grid(depth, rows, cols, stat):
    """Per-cell closeness, rescaled so 0 = farthest cell and 1 = nearest cell in this frame."""
    h, w = depth.shape
    reduce = np.max if stat == 'max' else np.mean
    grid = np.array([[reduce(depth[r * h // rows:(r + 1) * h // rows, c * w // cols:(c + 1) * w // cols])
                      for c in range(cols)] for r in range(rows)])
    return (grid - grid.min()) / (grid.max() - grid.min() + 1e-8)


def draw_grid(img, grid):
    """Draw cell borders and values, outlining the nearest cell red and the farthest cyan."""
    rows, cols = grid.shape
    h, w = img.shape[:2]
    cell = lambda r, c: ((c * w // cols, r * h // rows), ((c + 1) * w // cols, (r + 1) * h // rows))
    for r in range(rows):
        for c in range(cols):
            (x0, y0), (x1, y1) = cell(r, c)
            cv2.rectangle(img, (x0, y0), (x1, y1), (80, 80, 80), 1)
            text = f'{grid[r, c]:.1f}'
            cv2.putText(img, text, (x0 + 6, y0 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3)
            cv2.putText(img, text, (x0 + 6, y0 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    for idx, color in ((grid.argmax(), (0, 0, 255)), (grid.argmin(), (255, 255, 0))):
        cv2.rectangle(img, *cell(*np.unravel_index(idx, grid.shape)), color, 3)


def draw_marker(img, point, label, color):
    """Draw a crosshair with a filled label box at point (x, y)."""
    x, y = point
    cv2.circle(img, point, 12, (0, 0, 0), 4)
    cv2.circle(img, point, 12, color, 2)
    cv2.drawMarker(img, point, color, cv2.MARKER_CROSS, 30, 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    # Keep the label inside the image near the edges.
    tx = min(max(x + 15, 0), img.shape[1] - tw - 6)
    ty = min(max(y - 15, th + 6), img.shape[0] - 6)
    cv2.rectangle(img, (tx - 3, ty - th - 5), (tx + tw + 3, ty + 5), color, -1)
    cv2.putText(img, label, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

prev = time.time()
avg_depth = None    # running (time-averaged) depth map
search_mask = None  # 255 where NEAR/FAR may be found, 0 on the ignored border

print('Press q in the video window to quit.')
while True:
    ok, frame = cap.read()
    if not ok:
        print('Failed to read a frame from the webcam.')
        break

    depth = model.infer_image(frame, INPUT_SIZE).astype(np.float32)

    # Level 1: nearest and farthest points.
    # Depth Anything V2 outputs relative inverse depth, so higher = closer:
    # the max is the NEAREST point and the min is the FARTHEST.
    # 1. Blur so single noisy pixels can't win.
    # 2. Scale to 0-1: raw values aren't comparable between frames.
    # 3. Average over time so near-ties (e.g. a flat back wall) don't flip every frame.
    # 4. Search inside the mask, skipping the unreliable border.
    # NEAR = single closest pixel (keeps small close objects like a finger).
    # FAR = centre of the biggest far region (a flat wall has thousands of near-tied
    #       pixels, so the single farthest one hops around; the region's centre doesn't).
    smooth = cv2.GaussianBlur(depth, (BLUR_KSIZE, BLUR_KSIZE), 0)
    smooth = (smooth - smooth.min()) / (smooth.max() - smooth.min() + 1e-8)
    if avg_depth is None:
        avg_depth = smooth
        h, w = smooth.shape
        mx, my = int(w * EDGE_MARGIN), int(h * EDGE_MARGIN)
        search_mask = np.zeros((h, w), np.uint8)
        search_mask[my:h - my, mx:w - mx] = 255
    else:
        avg_depth = SMOOTH_ALPHA * smooth + (1 - SMOOTH_ALPHA) * avg_depth
    _, _, _, near_pt = cv2.minMaxLoc(avg_depth, search_mask)
    inside = search_mask > 0
    far_cut = np.percentile(avg_depth[inside], FAR_PERCENT)
    far_pt = region_center((avg_depth <= far_cut) & inside)

    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8) * 255.0
    depth = depth.astype(np.uint8)
    depth_color = (cmap(depth)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)

    now = time.time()
    fps = 1.0 / max(now - prev, 1e-6)
    prev = now
    cv2.putText(depth_color, f'{MODEL} {fps:.1f} FPS', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    if GRID_ROWS:
        draw_grid(frame, depth_grid(avg_depth, GRID_ROWS, GRID_COLS, GRID_STAT))

    for img in (frame, depth_color):
        draw_marker(img, near_pt, 'NEAR', (0, 0, 255))    # red (BGR)
        draw_marker(img, far_pt, 'FAR', (255, 255, 0))    # cyan (BGR)

    cv2.imshow('Depth Anything V2 - webcam (left) and depth (right). Press q to quit.',
               np.hstack([frame, depth_color]))
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## Step 4: Run it

This opens a window with your webcam on the left and the depth map on the right. By default, warm colors (red) are close to the camera and cool colors (blue) are far away. A red **NEAR** marker shows the closest point in the scene and a cyan **FAR** marker the farthest, on both the camera feed and the depth map. Click the video window and press **q** to quit. The cell keeps running until you do.

The cell uses the same Python as this notebook, so it works on both Mac and Windows.

In [ ]:
import sys

!{sys.executable} webcam_depth.py

## Step 5: Try it out

- Hold your hand close to the camera, then move it away. The colors should shift.
- Walk back from the camera and watch your body change color.
- Point the camera at a room to see walls, furniture and doorways at different depths.
- Depth is **relative**, not in meters. The colors show what is nearer or farther, not exact distance.

## Tuning speed

Edit the settings at the top of the script cell in Step 3, run that cell again, then re-run Step 4.

| Setting | Change | Effect |
|---|---|---|
| `ENCODER` | `vits` (fastest), `vitb`, `vitl` (slowest) | Larger models are more accurate but slower. The encoder must match the checkpoint you downloaded. |
| `INPUT_SIZE` | Lower (for example 252) or higher (518) | Lower is faster. 518 is the repo default. |
| Device | NVIDIA GPU (Windows) or Apple Silicon (Mac) | Much faster than CPU. The script picks the best one automatically. |

On a CPU-only machine, expect a low frame rate even with the Small model. That is normal.

## Troubleshooting

- **"Could not open the webcam":** check the permissions in Step 1 and close other apps using the camera. Try `CAMERA_INDEX = 1`.
- **Wrong camera opens (for example a phone via Continuity Camera on Mac):** change `CAMERA_INDEX` to 0 or 1.
- **Black or frozen window on Windows:** make sure the `cv2.CAP_DSHOW` line is in the script, or try removing it.
- **Very low FPS:** lower `INPUT_SIZE`, use `vits`, and close other heavy apps. On Windows with an NVIDIA GPU, confirm you installed the CUDA build of PyTorch. The script prints the device it uses at startup, and `cpu` there means no GPU is being used.
- **`FileNotFoundError` for the .pth file:** the checkpoint is not in `checkpoints/`, or `ENCODER` does not match the file name.
- **`ModuleNotFoundError: depth_anything_v2`:** the notebook is not inside the `Depth-Anything-V2` folder. Re-run Step 2.
- **OpenCV window does not appear on Mac:** if you installed `opencv-python-headless`, uninstall it and run `%pip install opencv-python`.
- **Notebook seems stuck after pressing q:** the cell stops when the video window closes. If it does not, use the notebook's Interrupt button.

## Links

- Code: https://github.com/DepthAnything/Depth-Anything-V2
- Models: https://huggingface.co/depth-anything